# 第1回 量子ビットとは何か

この回のゴール:
1. 古典ビット（ふつうのコンピュータのビット）と量子ビットの違いを言葉で説明できる
2. 「重ね合わせ」と「測定すると確率で結果が決まる」ことを、シミュレーターで何百回も測定して確かめる
3. ブロッホ球という図で、量子ビットの状態を「地球儀の上の点」として見られるようになる

進め方: 上のセルから順に `Shift + Enter` で実行してください。コードの数字を変えて何度も実行してみるのが一番の勉強です。

## 0. 準備

In [ ]:
# このセルは毎回最初に実行します（必要な道具を読み込む）
import numpy as np                      # 数値計算（ルートや三角関数など）
import matplotlib.pyplot as plt         # グラフを描く
from IPython.display import display     # 図を何枚も表示するため

import qiskit
import qiskit_aer
from qiskit import QuantumCircuit                       # 量子回路を作る
from qiskit.quantum_info import Statevector             # 量子ビットの「状態」を表す
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from qiskit_aer import AerSimulator                     # パソコンの中のシミュレーター
from qiskit_aer.primitives import SamplerV2             # 「測定して結果を数える」道具
from qiskit.transpiler import generate_preset_pass_manager

print("Qiskit:", qiskit.__version__, "/ Aer:", qiskit_aer.__version__)

`Qiskit: 2.…` と表示されればOKです。エラーが出たら README の「環境の準備」をもう一度確認してください。

次のセルでは、このノートで何度も使う「道具（関数）」を用意します。中身は今は分からなくて大丈夫です。
- `measure(状態, 回数)` … 量子ビットをその状態にして、指定回数だけ測定し、0と1がそれぞれ何回出たかを返す
- `probs(状態)` … その状態を測定したとき、0と1がそれぞれ出る確率を表示する
- `state_from_angle(角度)` … あとでブロッホ球のところで使う

In [ ]:
simulator = AerSimulator()
# 回路をシミュレーターが実行できる形に変換する「変換器」
pm = generate_preset_pass_manager(backend=simulator, optimization_level=1)
sampler = SamplerV2()


def measure(state, shots):
    """量子ビットを state の状態にして、shots 回 測定した結果を数えて返す"""
    qc = QuantumCircuit(1)
    qc.prepare_state(state, 0)   # 量子ビット0番を、指定した状態にする
    qc.measure_all()             # 測定する
    job = sampler.run([pm.run(qc)], shots=shots)
    return job.result()[0].data.meas.get_counts()


def probs(state):
    """測定したとき 0 と 1 が出る確率を、見やすく(小数3けたで)表示する"""
    return {str(k): round(float(v), 3) for k, v in state.probabilities_dict().items()}


def state_from_angle(theta_deg):
    """ブロッホ球の北極から theta_deg 度 傾けた状態を作る"""
    t = np.radians(theta_deg)
    return Statevector([np.cos(t / 2), np.sin(t / 2)])

## 1. 古典ビット: いつも「0か1のどちらか」

ふつうのコンピュータの情報の最小単位は **ビット** です。ビットは必ず `0` か `1` のどちらか一方で、
「いつ見ても同じ値」です。見たからといって値が変わることはありません。

| | 古典ビット | 量子ビット |
|---|---|---|
| とれる状態 | 0 か 1 のどちらか | 0 と 1 の **重ね合わせ** もとれる |
| 見る（測定する）と | いつも同じ値 | **確率で** 0 か 1 が出る |
| 見たあと | 変わらない | 出た値の状態に **変わってしまう** |

この表の右側を、これから1つずつシミュレーターで確かめていきます。

## 2. 量子ビットの状態は「2つの数の組」で表す

量子ビットの状態は、**2つの数を縦に並べたもの（ベクトル）** で表します。線形代数で習うベクトルそのものです。

- 「0の状態」: $|0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}$
- 「1の状態」: $|1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$

**記号の意味:** $|0\rangle$ は「ケット0」と読みます。「量子ビットの0の状態」という名前のラベルだと思ってください。
縦に並んだ2つの数のうち、**上の数は「0」に関係する数、下の数は「1」に関係する数** です。
$|0\rangle$ は「上が1、下が0」なので「完全に0の状態」、$|1\rangle$ は「完全に1の状態」です。

Qiskit では `Statevector([上の数, 下の数])` で状態を作れます。

In [ ]:
ket0 = Statevector([1, 0])   # |0> の状態
ket1 = Statevector([0, 1])   # |1> の状態

display(ket0.draw("latex"))
display(ket1.draw("latex"))

# 測定したとき、0 と 1 がそれぞれ出る確率
print("|0> を測定したときの確率:", probs(ket0))
print("|1> を測定したときの確率:", probs(ket1))

`{'0': 1.0}` は「測定すると 100% の確率で 0 が出る」という意味です。ここまでは古典ビットと同じですね。

## 3. 重ね合わせ: 0 でもあり 1 でもある状態

量子ビットは、上の数と下の数の **両方が0でない** 状態もとれます。これを **重ね合わせ** といいます。

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle = \begin{pmatrix} \alpha \\ \beta \end{pmatrix}$$

**この式の意味:**
- $|\psi\rangle$（ケット・プサイ）は「ある量子ビットの状態」につけた名前です。
- $\alpha$（アルファ）と $\beta$（ベータ）はただの数で、「0の成分がどれくらいか」「1の成分がどれくらいか」を表します。
- 「$|0\rangle$ を $\alpha$ 倍したものと、$|1\rangle$ を $\beta$ 倍したものを足した」という意味で、ベクトルで書くと右側の形になります。

そして、ここが一番大事なルールです:

> **測定すると、0 が出る確率は $\alpha^2$、1 が出る確率は $\beta^2$**

つまり「**数を2乗したものが確率**」です。確率は全部足すと 1（=100%）にならないといけないので、
$\alpha^2 + \beta^2 = 1$ という決まりがあります。
（※ 本当は $\alpha, \beta$ は複素数でもよく、そのときは「絶対値の2乗」が確率になります。この回では実数だけを使います。）

いちばん有名な重ね合わせは、$\alpha = \beta = \frac{1}{\sqrt{2}}$ の状態です。
$\left(\frac{1}{\sqrt{2}}\right)^2 = \frac{1}{2}$ なので、0 と 1 がちょうど半分ずつの確率で出ます。
この状態には $|+\rangle$（ケット・プラス）という名前がついています。

In [ ]:
plus = Statevector([1 / np.sqrt(2), 1 / np.sqrt(2)])   # |+> の状態

display(plus.draw("latex"))
print("確率:", probs(plus))

`0.7071...` は $\frac{1}{\sqrt{2}}$ のこと、確率は 0 も 1 も `0.5`（50%）になりました。

**確率の合計が 1 にならない状態は作れません。** 試しに `[1, 1]` を入れてみます（確率が 1+1=2 になってしまう）。
`is_valid()` は「量子ビットの状態として正しいか」を調べる関数です。

In [ ]:
print("[1/√2, 1/√2] は正しい状態?", plus.is_valid())
print("[1, 1]       は正しい状態?", Statevector([1, 1]).is_valid())

## 4. 測定: 結果は確率で決まる

では、$|+\rangle$ の量子ビットを実際に（シミュレーターで）測定してみましょう。まずは回路の図を見てみます。

In [ ]:
qc = QuantumCircuit(1)
qc.prepare_state(plus, 0)   # 量子ビットを |+> の状態にする
qc.measure_all()            # 測定する
qc.draw("mpl")

**図の読み方:** 左から右へ時間が進みます。
- `q` の横線 … 量子ビット1個
- `State Preparation` の箱 … 量子ビットを指定した状態にする
- 点線の仕切り … 「ここで区切る」という目印（バリア）。計算には影響しません
- メーターの絵 … 測定。結果（0か1）が下の線 `meas` に書き込まれる

では **1回だけ** 測定するのを、10回くり返してみます。

In [ ]:
for i in range(10):
    result = measure(plus, shots=1)   # shots=1 は「1回だけ測定」
    print(f"{i + 1}回目:", list(result.keys())[0])

0 と 1 がバラバラに出たはずです。もう一度このセルを実行すると、**違う並び** になります。
同じ状態を用意して同じように測定しているのに、結果は毎回変わる。どちらが出るかは前もって決まっておらず、
**確率でしか分からない** ——これが量子ビットの不思議な性質です。

1回ずつでは偶然に見えますが、たくさん測定すると確率が見えてきます。回数を 10、100、1000 と増やしてみましょう。

In [ ]:
for shots in [10, 100, 1000]:
    counts = measure(plus, shots=shots)
    print(f"{shots:5d}回 測定:", counts, " → 0の割合 =", round(counts.get("0", 0) / shots, 3))

In [ ]:
counts = measure(plus, shots=1000)
plot_histogram(counts, title="|+> measured 1000 times")

回数を増やすほど、0の割合が **0.5（=50%）に近づく** はずです。これが「確率 $\alpha^2 = 0.5$」の意味です。

※ グラフの文字は、日本語フォントがなくても文字化けしないように英語にしています。

### 確率が半々でない重ね合わせ

$\alpha$ と $\beta$ を変えれば、確率も変えられます。例えば「0 が 80%、1 が 20%」にしたいなら、
2乗して 0.8 と 0.2 になる数、つまり $\alpha = \sqrt{0.8},\ \beta = \sqrt{0.2}$ にすればよいです。

In [ ]:
biased = Statevector([np.sqrt(0.8), np.sqrt(0.2)])
print("理論上の確率:", probs(biased))

counts = measure(biased, shots=1000)
print("1000回測定した結果:", counts)
plot_histogram(counts, title="0: 80%, 1: 20%")

### 測定すると、状態が変わってしまう

古典ビットとのもう1つの大きな違いは、**測定すると状態が変わる** ことです。
$|+\rangle$ を測定して 0 が出たら、その瞬間に量子ビットは $|0\rangle$ になります（1 が出たら $|1\rangle$ になる）。
重ね合わせは壊れてしまうのです。

確かめてみましょう。同じ量子ビットを **続けて2回** 測定します。もし重ね合わせが残っているなら、
1回目と2回目はバラバラ（00, 01, 10, 11 が全部出る）になるはずです。

In [ ]:
qc2 = QuantumCircuit(1, 2)      # 量子ビット1個、結果を書く場所2個
qc2.prepare_state(plus, 0)
qc2.measure(0, 0)               # 1回目の測定 → 結果を0番に書く
qc2.measure(0, 1)               # 2回目の測定 → 結果を1番に書く
display(qc2.draw("mpl"))

counts2 = sampler.run([pm.run(qc2)], shots=1000).result()[0].data.c.get_counts()
print(counts2)

結果は `00` と `11` だけで、`01` や `10` は **1回も** 出ません（文字列は右が1回目、左が2回目です）。
1回目に 0 が出たら、2回目も必ず 0。1回目の測定で量子ビットが $|0\rangle$ か $|1\rangle$ に「確定」してしまったからです。

## 5. ブロッホ球: 量子ビットの状態を「地球儀の上の点」で見る

量子ビットの状態は、**ブロッホ球** という球の表面上の1点（矢印の先）として描くことができます。地球儀だと思ってください。

- **北極** … $|0\rangle$（必ず0が出る）
- **南極** … $|1\rangle$（必ず1が出る）
- **赤道** … 0 と 1 が半々で出る重ね合わせ（$|+\rangle$ など）
- 北半球の点ほど 0 が出やすく、南半球の点ほど 1 が出やすい

つまり **「北極と南極のどちらに近いか」が、測定の確率を表します。** まずは4つの状態を描いてみましょう。

In [ ]:
for name, state in [("|0>", ket0), ("|1>", ket1), ("|+>", plus), ("80% / 20%", biased)]:
    print(name, "の確率:", probs(state))
    display(plot_bloch_multivector(state, title=name, figsize=(3, 3)))

図の見方: 球の上が $|0\rangle$（北極）、下が $|1\rangle$（南極）です。矢印の向きが状態を表しています。
$|+\rangle$ は赤道（横向き）、80%/20% の状態は北半球（やや上向き）を指していますね。

### 北極からの角度と確率の関係

北極から角度 $\theta$（シータ）だけ傾けた状態を考えます。このとき

$$0\ \text{が出る確率} = \cos^2\frac{\theta}{2}$$

**この式の意味:** 「傾けた角度の半分のコサインを2乗したもの」が 0 の出る確率です。
- $\theta = 0^\circ$（北極）なら $\cos^2 0^\circ = 1$ → 必ず 0
- $\theta = 90^\circ$（赤道）なら $\cos^2 45^\circ = 0.5$ → 半々
- $\theta = 180^\circ$（南極）なら $\cos^2 90^\circ = 0$ → 必ず 1

式を覚える必要はありません。「**北極から離れるほど、なめらかに 0 が出にくくなる**」ことだけつかめれば十分です。
角度を変えながら、図と確率を見てみましょう（`state_from_angle(角度)` は準備のセルで作った道具です）。

In [ ]:
for deg in [0, 45, 90, 135, 180]:
    s = state_from_angle(deg)
    p0 = probs(s).get("0", 0)
    print(f"角度 {deg:3d}°: 0が出る確率 = {p0:.3f}")
    display(plot_bloch_multivector(s, title=f"theta = {deg} deg", figsize=(2.5, 2.5)))

### 理論と「実験」を比べる

最後に、角度を細かく変えながら **理論の確率（線）** と、**シミュレーターで1000回測定した割合（点）** を1枚のグラフにします。
点が線の上にのっていれば、「状態が確率を決めている」ことを実験で確かめたことになります。

In [ ]:
angles = np.arange(0, 181, 15)              # 0°, 15°, 30°, ..., 180°
theory = np.cos(np.radians(angles) / 2) ** 2 # 理論の確率

shots = 1000
experiment = []
for deg in angles:
    counts = measure(state_from_angle(deg), shots=shots)
    experiment.append(counts.get("0", 0) / shots)

fine = np.linspace(0, 180, 200)
plt.figure(figsize=(6, 4))
plt.plot(fine, np.cos(np.radians(fine) / 2) ** 2, label="theory: cos^2(theta/2)")
plt.plot(angles, experiment, "o", label=f"simulator ({shots} shots)")
plt.xlabel("angle from north pole theta [deg]")
plt.ylabel("probability of measuring 0")
plt.xticks(range(0, 181, 30))
plt.grid(alpha=0.3)
plt.legend()
plt.show()

### おまけ: 確率が同じでも、違う状態がある

$|+\rangle = \begin{pmatrix} 1/\sqrt{2} \\ 1/\sqrt{2} \end{pmatrix}$ の下の数の符号をマイナスにした
$|-\rangle = \begin{pmatrix} 1/\sqrt{2} \\ -1/\sqrt{2} \end{pmatrix}$（ケット・マイナス）を考えます。

2乗するとマイナスは消えるので、**測定の確率は $|+\rangle$ とまったく同じ 50% / 50%** です。
ところがブロッホ球では、赤道上の **反対側** を指します。

In [ ]:
minus = Statevector([1 / np.sqrt(2), -1 / np.sqrt(2)])
print("|+> の確率:", probs(plus))
print("|-> の確率:", probs(minus))
display(plot_bloch_multivector(plus, title="|+>", figsize=(3, 3)))
display(plot_bloch_multivector(minus, title="|->", figsize=(3, 3)))

測定するだけでは区別できないのに、別の状態。この「赤道のどちら向きか」という情報（**位相** といいます）は、
量子コンピュータが速く計算できる秘密のカギになります。次回の量子ゲートで、この違いを見分ける方法が出てきます。

## まとめ

- 量子ビットの状態は **2つの数の組（ベクトル）** $\begin{pmatrix} \alpha \\ \beta \end{pmatrix}$ で表す
- 両方が0でない状態を **重ね合わせ** という
- 測定すると、**0 が確率 $\alpha^2$、1 が確率 $\beta^2$** で出る。どちらが出るかは前もって決まっていない
- 測定すると、重ね合わせは壊れて、出た値の状態に変わる
- **ブロッホ球** では、北極が $|0\rangle$、南極が $|1\rangle$、赤道が半々の重ね合わせ。北極に近いほど 0 が出やすい
- 確率が同じでも、ブロッホ球の上で違う点になる状態（$|+\rangle$ と $|-\rangle$）がある

## 練習問題

解答例は `solutions/01_qubit_solutions.ipynb` にあります。まず自分でやってみてください。
下に空のセルを用意してあります。準備のセル（`measure` などを作ったセル）を実行してから使ってください。

### 問題1: 70% / 30% の状態を作る
「0 が 70%、1 が 30% の確率で出る」量子ビットの状態を `Statevector` で作り、
(a) `probs()` で確率を確認し、(b) 1000回測定してヒストグラムを描いてください。
ヒント: 2乗して 0.7 になる数は何でしょう？

In [ ]:
# 問題1


### 問題2: 確率を予想してから確かめる
状態 $\begin{pmatrix} 0.6 \\ 0.8 \end{pmatrix}$ について、
(a) 正しい量子ビットの状態になっているか（確率の合計が1か）を **手で計算** してから `is_valid()` で確かめる、
(b) 0 と 1 が出る確率を手で予想してから、2000回測定して予想と比べる、
(c) この状態はブロッホ球の北半球・南半球のどちらにあるか予想してから、図を描いて確かめる。

In [ ]:
# 問題2


### 問題3: 測定回数と「ばらつき」
$|+\rangle$ を **100回** 測定して「0の割合」を求める実験を5回くり返してください。
次に、**10000回** 測定する実験を5回くり返してください。
5回の結果のばらつき（いちばん大きい値といちばん小さい値の差）は、100回と10000回でどう違いますか？
そこから「確率を正確に知りたいとき、測定回数をどうするべきか」を言葉で書いてください。

In [ ]:
# 問題3
